In [27]:
!pip install -qqq nltk

import nltk
import re
from collections import Counter

In [28]:
nltk.download('words', quiet=True)
nltk.download('punkt', quiet=True)

print("NLTK resources downloaded successfully.")

NLTK resources downloaded successfully.


In [29]:
text = (
    "Formula 1 is the pinnacle of motorsport. "
    "Lewis Hamilton is a seven-time world champion. "
    "Red Bull Racing and Mercedes are fierce rivals. "
    "The Grand Prix races are held on circuits all over the world. "
    "Drivers like Max Verstappen, Charles Leclerc, and George Russell "
    "compete for victory. "
    "The pit stop strategy is crucial for success. "
    "The cars are incredibly fast, reaching speeds over 300 km/h. "
    "Tyre management and aerodynamic efficiency are key factors. "
    "Ferrari and McLaren are famous Formula 1 teams. "
    "Racing drivers compete for the Drivers Championship. "
    "Teams compete for the Constructors Championship. "
    "Qualifying determines the starting grid. "
    "Pit stops are important for changing tyres. "
    "Race strategy can determine the final result. "
    "Mercedes, Ferrari, Red Bull, and McLaren have many victories."
) * 1000

print("F1 corpus created successfully.")

F1 corpus created successfully.


In [30]:
words = re.findall(r'\b[a-z]+\b', text.lower())

word_counts = Counter(words)
total_words = sum(word_counts.values())

print("Total words:", total_words)
print("Unique words:", len(word_counts))

Total words: 120000
Unique words: 80


In [31]:
def P(word):
    return word_counts[word] / total_words if total_words > 0 else 0

print("Probability of 'racing':", P('racing'))
print("Probability of 'formula':", P('formula'))
print("Probability of 'ferrari':", P('ferrari'))
print("Probability of 'hamilton':", P('hamilton'))

Probability of 'racing': 0.016666666666666666
Probability of 'formula': 0.016666666666666666
Probability of 'ferrari': 0.016666666666666666
Probability of 'hamilton': 0.008333333333333333


In [32]:
alphabet = 'abcdefghijklmnopqrstuvwxyz'

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]

    deletes = [
        L + R[1:]
        for L, R in splits
        if R
    ]

    transposes = [
        L + R[1] + R[0] + R[2:]
        for L, R in splits
        if len(R) > 1
    ]

    replaces = [
        L + c + R[1:]
        for L, R in splits
        if R
        for c in alphabet
    ]

    inserts = [
        L + c + R
        for L, R in splits
        for c in alphabet
    ]

    return set(deletes + transposes + replaces + inserts)


def edits2(word):
    return set(
        e2
        for e1 in edits1(word)
        for e2 in edits1(e1)
    )

print("Edit candidates for 'raceng':")
print(list(edits1('raceng'))[:20])

Edit candidates for 'raceng':
['rachng', 'racegng', 'racengx', 'racmng', 'raoeng', 'raceag', 'aceng', 'racesng', 'baceng', 'racenf', 'rafeng', 'raceing', 'racevng', 'yraceng', 'rqceng', 'zraceng', 'nraceng', 'racenb', 'racehng', 'aaceng']


In [33]:
def known(words):
    return set(
        word for word in words
        if word in word_counts
    )


def candidates(word):
    return (
        known([word])
        or known(edits1(word))
        or known(edits2(word))
        or {word}
    )

print("Candidates for 'raceng':")
print(candidates('raceng'))

Candidates for 'raceng':
{'racing'}


In [34]:
def edit_distance(a, b):
    rows = len(a) + 1
    cols = len(b) + 1

    dp = [[0] * cols for _ in range(rows)]

    for i in range(rows):
        dp[i][0] = i

    for j in range(cols):
        dp[0][j] = j

    for i in range(1, rows):
        for j in range(1, cols):
            if a[i - 1] == b[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[-1][-1]


def channel_probability(noisy_word, candidate):
    distance = edit_distance(noisy_word, candidate)

    # Higher probability for fewer spelling errors
    return 1 / (2 ** distance)


print("Edit distance:", edit_distance("raceng", "racing"))
print("Channel probability:",
      channel_probability("raceng", "racing"))

Edit distance: 1
Channel probability: 0.5


In [35]:
def noisy_channel_score(noisy_word, candidate):
    language_probability = P(candidate)

    channel_probability_value = channel_probability(
        noisy_word,
        candidate
    )

    return (
        language_probability *
        channel_probability_value
    )


print(
    "Score for racing:",
    noisy_channel_score("raceng", "racing")
)

Score for racing: 0.008333333333333333


In [36]:
def correction(word):
    possible_candidates = candidates(word)

    best_candidate = max(
        possible_candidates,
        key=lambda candidate: noisy_channel_score(
            word,
            candidate
        )
    )

    return best_candidate


print("Correction for 'raceng':", correction("raceng"))
print("Correction for 'hamiltonn':", correction("hamiltonn"))
print("Correction for 'versstappen':", correction("versstappen"))
print("Correction for 'merceds':", correction("merceds"))

Correction for 'raceng': racing
Correction for 'hamiltonn': hamilton
Correction for 'versstappen': verstappen
Correction for 'merceds': mercedes


In [37]:
def show_candidates(word):
    possible_candidates = candidates(word)

    results = []

    for candidate in possible_candidates:
        language_prob = P(candidate)

        channel_prob = channel_probability(
            word,
            candidate
        )

        score = language_prob * channel_prob

        results.append(
            (
                candidate,
                language_prob,
                channel_prob,
                score
            )
        )

    results.sort(key=lambda x: x[3], reverse=True)

    print(f"\nNoisy word: {word}")
    print("-" * 65)
    print(
        f"{'Candidate':15}"
        f"{'P(candidate)':18}"
        f"{'P(error)':15}"
        f"{'Final Score':15}"
    )
    print("-" * 65)

    for candidate, lp, cp, score in results:
        print(
            f"{candidate:15}"
            f"{lp:<18.6f}"
            f"{cp:<15.6f}"
            f"{score:<15.8f}"
        )

    print("-" * 65)
    print("Best correction:", results[0][0])

In [38]:
show_candidates("raceng")


Noisy word: raceng
-----------------------------------------------------------------
Candidate      P(candidate)      P(error)       Final Score    
-----------------------------------------------------------------
racing         0.016667          0.500000       0.00833333     
-----------------------------------------------------------------
Best correction: racing


In [39]:
test_words = [
    "raceng",
    "hamiltonn",
    "versstappen",
    "merceds",
    "ferrarri",
    "champiion"
]

print("NOISY CHANNEL SPELLING CORRECTION")
print("=" * 50)

for word in test_words:
    print(
        f"{word:15} -> {correction(word)}"
    )

NOISY CHANNEL SPELLING CORRECTION
raceng          -> racing
hamiltonn       -> hamilton
versstappen     -> verstappen
merceds         -> mercedes
ferrarri        -> ferrari
champiion       -> champion


In [40]:
sentence = (
    "hamiltonn is a world champiion "
    "and raceng for merceds"
)

sentence_words = re.findall(
    r'\b[a-z]+\b',
    sentence.lower()
)

corrected_words = []

for word in sentence_words:
    corrected_words.append(
        correction(word)
    )

corrected_sentence = " ".join(corrected_words)

print("Original sentence:")
print(sentence)

print("\nCorrected sentence:")
print(corrected_sentence)

Original sentence:
hamiltonn is a world champiion and raceng for merceds

Corrected sentence:
hamilton is a world champion and racing for mercedes


In [41]:
print("WORD-BY-WORD CORRECTION")
print("=" * 40)

for original, corrected in zip(
    sentence_words,
    corrected_words
):
    print(
        f"{original:15} -> {corrected}"
    )

WORD-BY-WORD CORRECTION
hamiltonn       -> hamilton
is              -> is
a               -> a
world           -> world
champiion       -> champion
and             -> and
raceng          -> racing
for             -> for
merceds         -> mercedes


In [42]:
correct_words = [
    "racing",
    "formula",
    "ferrari",
    "hamilton",
    "mercedes",
    "racing",
    "champion"
]

print("CORRECT WORD TEST")
print("=" * 40)

for word in correct_words:
    print(
        f"{word:15} -> {correction(word)}"
    )

CORRECT WORD TEST
racing          -> racing
formula         -> formula
ferrari         -> ferrari
hamilton        -> hamilton
mercedes        -> mercedes
racing          -> racing
champion        -> champion


In [43]:
print("TOP 15 F1 WORDS")
print("=" * 30)

for word, count in word_counts.most_common(15):
    print(
        f"{word:15} : {count}"
    )

TOP 15 F1 WORDS
the             : 9000
are             : 6000
and             : 5000
for             : 5000
is              : 3000
drivers         : 3000
compete         : 3000
formula         : 2000
world           : 2000
red             : 2000
bull            : 2000
racing          : 2000
mercedes        : 2000
over            : 2000
pit             : 2000
